In [4]:
import pandas as pd
dataset = pd.read_csv('C:\\Users\\Enass Irfan\\Downloads\\creditcard.csv')
dataset

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


# Data Preprocessing

In [5]:
dataset.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [6]:
X = dataset.drop('Class',axis = 1)
y = dataset['Class']

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

# Handle Imbalanced data using SMOTE

In [8]:
# this shows the original distrubution of non-fraudulent and fraudulent transactions
print('\n Original class distribution: 0-> Non-fraudulent   1-> Fraudulent')
print(y_train.value_counts())


 Original class distribution: 0-> Non-fraudulent   1-> Fraudulent
0    199008
1       356
Name: Class, dtype: int64


In [9]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state = 42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("\n Class distribution after applying SMOTE")
print(pd.Series(y_train_resampled).value_counts())


 Class distribution after applying SMOTE
0    199008
1    199008
Name: Class, dtype: int64


# Model Training

In [10]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state = 42)
model.fit(X_train_resampled, y_train_resampled)

RandomForestClassifier(random_state=42)

# Model Evaluation

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, roc_auc_score
y_pred = model.predict(X_test)
print('\nModel Evaluation')
print(classification_report(y_test, y_pred))
print(f"Precision: {precision_score(y_test,y_pred):.4f}")
print(f"Recall: {recall_score(y_test,y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test,y_pred):.4f}")


Model Evaluation
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85307
           1       0.83      0.88      0.85       136

    accuracy                           1.00     85443
   macro avg       0.92      0.94      0.93     85443
weighted avg       1.00      1.00      1.00     85443

Precision: 0.8322
Recall: 0.8750
F1-Score: 0.8530


# Interface

In [12]:
def test_fraud_detection(model, sample_data):
    try:
        sample_ds = pd.DataFrame([sample_data])
        sample_ds = sample_ds[X_train.columns]
        
        prediction = model.predict(sample_ds)
        prob = model.predict_proba(sample_ds)[:,1][0]
        
        if prediction[0] == 0:
            print('Transaction classified as NON-Fraudulent')
        else:
            print('Transaction is classified as Fraudulent')
        print(f"Probability of being fraudulent:{prob:.4f}")
    except Exception as e:
        print(f"Error: An error occured during testing:{e}")
        print("Please try again!")
        
feature_names = X_train.columns.tolist()

sample_data = {feature: 0 for feature in feature_names}


In [ ]:
while True:
    print('\n*** Fraud Detection Interface***')
    print("Enter transaction details. Type 'exit' to quit")
    
    for feature in feature_names:
        while True:
            try:
                value = input(f"Enter value for {feature}:")
                if value.lower() == 'exit':
                    break
                sample_data[feature] = float(value)
                break
            except ValueError:
                print('Invalid Input. Please try again')
        if value.lower() == 'exit':
            break
    if value.lower() != 'exit':
        test_fraud_detection(model, sample_data)


*** Fraud Detection Interface***
Enter transaction details. Type 'exit' to quit
Enter value for Time:1200
Enter value for V1:-1.0
Enter value for V2:1.0
Enter value for V3:-1.5
Enter value for V4:1.5
Enter value for V5:-2.0
Enter value for V6:2.0
Enter value for V7:-2.5
Enter value for V8:2.5
Enter value for V9:-3.0
Enter value for V10:3.0
Enter value for V11:-3.5
Enter value for V12:3.5
Enter value for V13:-4.0
Enter value for V14:4.0
Enter value for V15:-4.5
Enter value for V16:4.5
Enter value for V17:-5.0
Enter value for V18:5.0
Enter value for V19:-5.5
Enter value for V20:5.5
Enter value for V21:-6.0
Enter value for V22:6.0
Enter value for V23:-6.5
Enter value for V24:6.5
Enter value for V25:-7.0
Enter value for V26:7.0
Enter value for V27:-7.5
Enter value for V28:7.5
Enter value for Amount:25.00
Transaction classified as NON-Fraudulent
Probability of being fraudulent:0.1000

*** Fraud Detection Interface***
Enter transaction details. Type 'exit' to quit
Enter value for Time:exit
